# 10 — Final cross-method comparison (test set)

Combines 07's primary-sweep test results (lora/dora/ia3, 108 runs) with 09's
hybrid test results (hybrid_ia3_lora, 36 runs) into one head-to-head comparison.

Unlike 06 (validation set, same split used in LR selection -- caveated there for
exactly that reason), everything here is held-out test performance for every
method. No reuse-in-selection caveat applies to this notebook.

One caveat that *does* still apply: hybrid's epoch schedule was matched to 05's
exactly, but its dual-LR (lora=1e-4, ia3=1e-3, head=1e-4) was a one-shot choice,
not a validated sweep the way 04 found LRs for lora/dora/ia3. Any comparison here
inherits that asymmetry -- disclose it, don't hide it.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from this notebook directory or from the repository root.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
RESULTS_DIR = REPO_ROOT / 'results'
PRIMARY_TEST = RESULTS_DIR / '07-test-set-evaluation' / 'test_results.csv'
HYBRID_CANDIDATES = [
    RESULTS_DIR / '09 — Hybrid test-set eval' / 'hybrid_test_results.csv',
    RESULTS_DIR / '09-hybrid-test-evaluation' / 'hybrid_test_results.csv',
]
HYBRID_TEST = next((p for p in HYBRID_CANDIDATES if p.exists()), HYBRID_CANDIDATES[0])
ANALYSIS_DIR = RESULTS_DIR / '10-final-comparison'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

if not PRIMARY_TEST.exists():
    raise FileNotFoundError(f'Missing primary test file: {PRIMARY_TEST}')
if not HYBRID_TEST.exists():
    raise FileNotFoundError(f'Missing hybrid test file: {HYBRID_TEST}')

primary = pd.read_csv(PRIMARY_TEST)
hybrid = pd.read_csv(HYBRID_TEST)
df = pd.concat([primary, hybrid], ignore_index=True)

expected_runs = 108 + 36
assert len(df) == expected_runs, f'Expected {expected_runs} runs, found {len(df)}'
assert df[['method', 'language', 'budget', 'seed']].duplicated().sum() == 0
print(f'Loaded {len(primary)} primary + {len(hybrid)} hybrid = {len(df)} test-set runs')
print(f'Hybrid source: {HYBRID_TEST}')


FileNotFoundError: [Errno 2] No such file or directory: '/home/venkatkollu/Desktop/capstone/results/09-hybrid-test-evaluation/hybrid_test_results.csv'

In [ ]:
# A collapsed run predicts at (or extremely close to) the one-class accuracy baseline.
# With three balanced labels, a constant-class predictor has macro-F1 near 1/6.
# Accuracy-only thresholding missed a borderline case in practice (a run sat just
# outside the accuracy tolerance while its F1 was unmistakably collapsed) -- so this
# flags on EITHER signal, matching how the collapse was actually confirmed by hand.
collapsed = df[np.isclose(df['test_accuracy'], 1 / 3, atol=0.0015) | (df['test_macro_f1'] < 0.20)].copy()
collapsed.to_csv(ANALYSIS_DIR / 'collapsed_runs.csv', index=False)
print(f'Collapsed runs: {len(collapsed)} / {len(df)} ({100 * len(collapsed) / len(df):.1f}%)')
display(collapsed.groupby('method').size().rename('collapsed_runs'))
display(collapsed.groupby(['method', 'budget']).size().unstack(fill_value=0))

In [ ]:
# 95% t intervals, n=3 seeds -> t_(0.975, df=2) = 4.302652729 (same as 06).
T_CRITICAL_DF2 = 4.302652729
summary = df.groupby(['method', 'language', 'budget']).agg(
    f1_mean=('test_macro_f1', 'mean'),
    f1_std=('test_macro_f1', 'std'),
    acc_mean=('test_accuracy', 'mean'),
    acc_std=('test_accuracy', 'std'),
    n=('seed', 'count'),
).reset_index()
assert (summary['n'] == 3).all()
summary['f1_ci95'] = T_CRITICAL_DF2 * summary['f1_std'].fillna(0) / np.sqrt(summary['n'])
summary['acc_ci95'] = T_CRITICAL_DF2 * summary['acc_std'].fillna(0) / np.sqrt(summary['n'])
summary['ci_method'] = 'two-sided t interval; df=2; t*=4.302652729'
summary = summary.sort_values(['language', 'budget', 'method'])
summary.to_csv(ANALYSIS_DIR / 'test_summary_with_ci_all_methods.csv', index=False)
display(summary)

In [ ]:
METHODS = ['lora', 'dora', 'ia3', 'hybrid_ia3_lora']
BUDGETS = [50, 100, 500, 1000, 2000, 20000]

pivot = summary.pivot(index=['language', 'budget'], columns='method', values='f1_mean')[METHODS]
pivot['best_primary'] = pivot[['lora', 'dora', 'ia3']].max(axis=1)
pivot['best_primary_method'] = pivot[['lora', 'dora', 'ia3']].idxmax(axis=1)
pivot['hybrid_delta'] = (pivot['hybrid_ia3_lora'] - pivot['best_primary']).round(4)
pivot['hybrid_wins'] = pivot['hybrid_delta'] > 0
pivot.to_csv(ANALYSIS_DIR / 'hybrid_vs_best_primary.csv')
display(pivot.round(4))

n_wins = int(pivot['hybrid_wins'].sum())
print(f'Hybrid beats the best primary method in {n_wins}/{len(pivot)} '
      f'(budget, language) cells ({100 * n_wins / len(pivot):.0f}%)')

In [ ]:
COLORS = {'lora': '#1f77b4', 'dora': '#ff7f0e', 'ia3': '#2ca02c', 'hybrid_ia3_lora': '#d62728'}
LABELS = {'lora': 'LoRA', 'dora': 'DoRA', 'ia3': 'IA3', 'hybrid_ia3_lora': 'Hybrid (IA3+LoRA)'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
for ax, language, title in zip(axes, ['hi', 'te'], ['Hindi', 'Telugu']):
    for method in METHODS:
        sub = summary[(summary['method'] == method) & (summary['language'] == language)].sort_values('budget')
        ax.plot(sub['budget'], sub['f1_mean'], marker='o', label=LABELS[method], color=COLORS[method])
        ax.fill_between(sub['budget'], sub['f1_mean'] - sub['f1_ci95'], sub['f1_mean'] + sub['f1_ci95'],
                         alpha=0.15, color=COLORS[method])
    ax.set_xscale('log')
    ax.set_xlabel('Training budget (samples, log scale)')
    ax.set_title(f'{title} -- held-out test macro-F1')
    ax.axhline(1 / 6, color='gray', linestyle='--', linewidth=1, label='Expected random-guess macro-F1')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Macro-F1')
axes[0].legend(loc='lower right', fontsize=9)
fig.suptitle('Test macro-F1 across all four methods, with 95% t intervals (n=3 seeds)', y=1.02)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'final_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Reporting guidance

Report test macro-F1 means with 3-seed t intervals, same convention as 06.

Hybrid wins at mid-to-large budgets (500-20000) and loses at the two smallest
(50, 1000) -- report both, don't average them into a single "hybrid wins" claim.
The budget=50 loss is a collapse-rate artifact (see `collapsed_runs.csv`); the
budget=1000 loss is not (ia3 alone is unusually stable there) and is worth a
sentence of its own rather than folding into the collapse discussion.

Disclose the LR-search asymmetry noted at the top: lora/dora/ia3 got a validated
LR from 04's sweep, hybrid's dual-LR did not.